# Feature Engineering

## Ключевые инсайты из EDA:
- `phone_voip_call_state` = **25x lift** — главный сигнал фрода
- `event_type_nm` 12 (17x), 15 (12x), 10 (5.6x) — высокорисковые типы
- null в `mcc_code` у 69% фрода → is_null_mcc = мощный признак
- медиана суммы фрода (362k) vs нормальных (71k) — 5x разница
- `operaton_amt` null у 41% фрода → is_null_amount тоже признак
- `event_dttm` — строка, парсим вручную
- `compromised`, `developer_tools` — строки '0'/'1', кастуем в int

## Стратегия признаков (no leakage — closed='left' в rolling)

| Группа | Признаки | Lift |
|--------|----------|------|
| **VoIP / Security** | phone_voip_call_state, web_rdp, compromised | 25x / 0.4x / 0.2x |
| **Event type** | event_type_nm (12,15,10), is_high_risk_type | 17x |
| **Null flags** | is_null_mcc, is_null_amount | сильный |
| **Velocity** | cnt/amt за 1h/6h/24h/7d/30d | средний |
| **Amount** | log_amount, is_null_amount | средний |
| **Novelty** | new mcc/channel/currency для клиента | средний |
| **Session** | ops в сессии до текущей | средний |
| **Time** | hour, weekday, is_night | слабый |

In [ ]:
import polars as pl
import numpy as np
from pathlib import Path
import gc

ROOT = Path('/home/vadim/PyPr/hak')
PRETRAIN_TRAIN = ROOT / 'Pre-train_Train'
PRETEST_TEST = ROOT / 'Pre-test_Test'
DATA = ROOT / 'main_data'
FEATURES_OUT = ROOT / 'features'
FEATURES_OUT.mkdir(exist_ok=True)

WINDOWS_SEC = {
    '1h':  '1h',
    '6h':  '6h',
    '24h': '24h',
    '7d':  '7d',
    '30d': '30d',
}

# High-risk event types (из EDA)
HIGH_RISK_TYPES = [12, 15, 10, 6, 3]

print('Ready')

## 1. Загрузка и базовая обработка

In [ ]:
def load_and_clean(paths: list) -> pl.DataFrame:
    """Загружаем несколько parquet файлов и базово очищаем."""
    dfs = []
    for p in paths:
        print(f'  Loading {Path(p).name}...')
        df = pl.read_parquet(p)
        dfs.append(df)
    df = pl.concat(dfs)
    del dfs
    gc.collect()
    
    # Парсим дату (строка → datetime)
    df = df.with_columns(
        pl.col('event_dttm')
          .str.to_datetime('%Y-%m-%d %H:%M:%S')
          .alias('event_dttm')
    )
    
    # Строковые security флаги → int
    for col in ['compromised', 'developer_tools']:
        df = df.with_columns(
            pl.col(col).cast(pl.Int8, strict=False).fill_null(0).alias(col)
        )
    
    # mcc_code: строка → int (null остаётся null)
    df = df.with_columns(
        pl.col('mcc_code').cast(pl.Int32, strict=False).alias('mcc_code')
    )
    
    # battery: строка → float
    df = df.with_columns(
        pl.col('battery').cast(pl.Float32, strict=False).alias('battery')
    )
    
    # Сортируем по клиенту и времени
    df = df.sort(['customer_id', 'event_dttm'])
    
    print(f'Loaded: {df.shape}, memory: {df.estimated_size("mb"):.0f} MB')
    return df

## 2. Базовые признаки (из одной строки)

In [ ]:
def add_base_features(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        # Время
        pl.col('event_dttm').dt.hour().cast(pl.Int8).alias('hour'),
        pl.col('event_dttm').dt.weekday().cast(pl.Int8).alias('weekday'),
        pl.col('event_dttm').dt.month().cast(pl.Int8).alias('month'),
        (pl.col('event_dttm').dt.hour() < 6).cast(pl.Int8).alias('is_night'),
        
        # Сумма
        pl.col('operaton_amt').log1p().alias('log_amount'),
        pl.col('operaton_amt').is_null().cast(pl.Int8).alias('is_null_amount'),
        pl.col('operaton_amt').fill_null(0.0).alias('operaton_amt'),
        
        # Security flags (уже int после load_and_clean)
        pl.col('compromised').fill_null(0),
        pl.col('web_rdp_connection').fill_null(0),
        pl.col('phone_voip_call_state').fill_null(0),
        pl.col('developer_tools').fill_null(0),
        
        # Сумма всех флагов безопасности
        (
            pl.col('compromised').fill_null(0) +
            pl.col('web_rdp_connection').fill_null(0) +
            pl.col('phone_voip_call_state').fill_null(0) +
            pl.col('developer_tools').fill_null(0)
        ).cast(pl.Int8).alias('security_flags_sum'),
        
        # MCC: null = высокий риск (69% фрода имеет null mcc)
        pl.col('mcc_code').is_null().cast(pl.Int8).alias('is_null_mcc'),
        pl.col('mcc_code').fill_null(-1),
        
        # High-risk event types (17x и 12x lift)
        pl.col('event_type_nm').is_in(HIGH_RISK_TYPES).cast(pl.Int8).alias('is_high_risk_type'),
        
        # Battery аномалии (null → -1)
        pl.col('battery').fill_null(-1.0),
    ])

## 3. Velocity features (временные окна по клиенту)

In [ ]:
def add_velocity_features(df: pl.DataFrame) -> pl.DataFrame:
    """
    Rolling count/sum per customer, closed='left' → no leakage.
    Работаем с уже отсортированным df (customer_id, event_dttm).
    """
    new_cols = []
    
    for wname, wsec in WINDOWS_SEC.items():
        # Кол-во операций клиента за последние N (не включая текущую)
        new_cols.append(
            pl.col('operaton_amt')
            .rolling_sum(
                window_size=wsec,
                by='event_dttm',
                closed='left',
            )
            .over('customer_id')
            .alias(f'amt_sum_{wname}')
        )
        new_cols.append(
            pl.col('operaton_amt')
            .rolling_count(
                window_size=wsec,
                by='event_dttm',
                closed='left',
            )
            .over('customer_id')
            .cast(pl.Int32)
            .alias(f'cnt_{wname}')
        )
    
    # Время с предыдущей операции (секунды)
    new_cols.append(
        (
            pl.col('event_dttm').dt.timestamp('s') -
            pl.col('event_dttm').dt.timestamp('s').shift(1).over('customer_id')
        ).alias('secs_since_last')
    )
    
    # Кол-во voip операций за 24h
    new_cols.append(
        pl.col('phone_voip_call_state')
        .rolling_sum(
            window_size='24h',
            by='event_dttm',
            closed='left',
        )
        .over('customer_id')
        .alias('voip_cnt_24h')
    )
    
    return df.with_columns(new_cols)

## 4. Novelty features

In [ ]:
def add_novelty_features(df: pl.DataFrame) -> pl.DataFrame:
    """Новый ли MCC/канал для этого клиента (первый раз встречаем)."""
    new_cols = []
    for col in ['mcc_code', 'channel_indicator_type', 'currency_iso_cd']:
        new_cols.append(
            (pl.col(col).cum_count().over(['customer_id', col]) == 1)
            .cast(pl.Int8)
            .alias(f'is_new_{col}')
        )
    # Кол-во уникальных MCC за всё время до текущей операции
    new_cols.append(
        pl.col('mcc_code')
        .cum_count()
        .over('customer_id')
        .alias('cum_unique_mcc_approx')
    )
    return df.with_columns(new_cols)

## 5. Session features

In [ ]:
def add_session_features(df: pl.DataFrame) -> pl.DataFrame:
    """Кол-во и сумма операций в текущей сессии ДО текущей."""
    return df.with_columns([
        # Порядковый номер в сессии (0-based)
        (pl.col('event_id').cum_count().over('session_id') - 1)
          .cast(pl.Int32)
          .alias('session_ops_before'),
        # Сумма в сессии до текущей
        (pl.col('operaton_amt').cum_sum().over('session_id') - pl.col('operaton_amt'))
          .alias('session_amt_before'),
    ])

## 6. Полный pipeline

In [ ]:
def build_features(df: pl.DataFrame) -> pl.DataFrame:
    print('  base features...')
    df = add_base_features(df)
    print('  velocity features...')
    df = add_velocity_features(df)
    print('  novelty features...')
    df = add_novelty_features(df)
    print('  session features...')
    df = add_session_features(df)
    return df


FEATURE_COLS = [
    # Time
    'hour', 'weekday', 'month', 'is_night',
    # Amount
    'log_amount', 'operaton_amt', 'is_null_amount',
    # Security (phone_voip = TOP)
    'phone_voip_call_state', 'web_rdp_connection', 'compromised',
    'developer_tools', 'security_flags_sum',
    # Event type
    'event_type_nm', 'is_high_risk_type',
    # MCC
    'mcc_code', 'is_null_mcc',
    # Channel / currency
    'channel_indicator_type', 'channel_indicator_sub_type', 'currency_iso_cd',
    # Device
    'battery', 'operating_system_type', 'pos_cd',
    # Velocity
    'cnt_1h', 'cnt_6h', 'cnt_24h', 'cnt_7d', 'cnt_30d',
    'amt_sum_1h', 'amt_sum_6h', 'amt_sum_24h', 'amt_sum_7d', 'amt_sum_30d',
    'secs_since_last', 'voip_cnt_24h',
    # Novelty
    'is_new_mcc_code', 'is_new_channel_indicator_type', 'is_new_currency_iso_cd',
    'cum_unique_mcc_approx',
    # Session
    'session_ops_before', 'session_amt_before',
    # Timezone (поведенческий паттерн)
    'timezone',
]

## 7. Запуск: строим фичи для TRAIN

In [ ]:
%%time

# Загружаем ВСЁ: pretrain (история) + train (разметка)
# Pretrain нужен как история для velocity/novelty признаков
print('Loading pretrain + train...')
all_files = sorted(PRETRAIN_TRAIN.glob('*.parquet'))
print('Files:', [f.name for f in all_files])

df_all = load_and_clean(all_files)
print(f'Total rows: {len(df_all):,}')

In [ ]:
%%time

print('Building features...')
df_all = build_features(df_all)
print(f'Done. Memory: {df_all.estimated_size("mb"):.0f} MB')

In [ ]:
%%time

# Оставляем только train период + присоединяем метки
labels = pl.read_parquet(DATA / 'train_labels.parquet')

# Train период: 2024-10-01 → 2025-05-31
df_train = df_all.filter(
    pl.col('event_dttm') >= pl.lit('2024-10-01').str.to_datetime('%Y-%m-%d')
)

# Присоединяем метки (только labeled строки имеют target)
df_train = df_train.join(
    labels.select(['event_id', 'target']),
    on='event_id',
    how='left'
)

print(f'Train rows: {len(df_train):,}')
print('Target distribution:')
print(df_train['target'].value_counts(sort=True))
print(f'  Null (green): {df_train["target"].is_null().sum():,}')

# Сохраняем
df_train.write_parquet(FEATURES_OUT / 'train_features.parquet')
print('Saved train_features.parquet')

## 8. Запуск: строим фичи для TEST

In [ ]:
%%time

# Для теста нужна полная история: pretrain + train + pretest
# Тестовые строки (test.parquet) — последний день каждого клиента
print('Loading pretest + test...')
df_pretest = load_and_clean([PRETEST_TEST / 'pretest.parquet'])
df_test_raw = load_and_clean([PRETEST_TEST / 'test.parquet'])

# Объединяем ВСЮ историю + тест строки
print('Combining all history + test...')
df_full = pl.concat([df_all, df_pretest, df_test_raw]).sort(['customer_id', 'event_dttm'])
del df_all, df_pretest
gc.collect()

print(f'Full dataset: {len(df_full):,} rows, {df_full.estimated_size("mb"):.0f} MB')

In [ ]:
%%time

print('Building features for test...')
df_full = build_features(df_full)

# Берём только тестовые event_id
test_ids = set(df_test_raw['event_id'].to_list())
df_test_feat = df_full.filter(pl.col('event_id').is_in(test_ids))

print(f'Test features: {len(df_test_feat):,} rows')
df_test_feat.write_parquet(FEATURES_OUT / 'test_features.parquet')
print('Saved test_features.parquet')